In [1]:
import sys
sys.path.append("..")  # Adds the parent directory (src) to the Python path

In [2]:
from backend.backend.utils import pod_parser

In [3]:
import requests
SERVER_URL = "http://127.0.0.1:8008"
# gather the podcast slug, episode guid, and audio file link for transcription
podcasts = requests.get(f"{SERVER_URL}/api/podcasts/")
podcasts = podcasts.json()

In [4]:
def get_new_episodes(podcast):
    """Get the new episodes of a podcast from the RSS feed.

    Parameters
    ----------
    podcast : dict
        The podcast dictionary.

    Returns
    -------
    list
        A list of new episodes.
    """
    episodes_all = pod_parser.parse_channel(podcast["rss"])
    episodes_db = [entry["guid"] for entry in podcast["audioitem_set"]]

    # find the highest index in episodes_all that has a guid matching any guid in db_episodes
    #inefficient, but works
    idx = 0
    for i, episode in enumerate(episodes_all["audioitem_set"]):
        if episode["guid"] in episodes_db:
            idx = i

    # with the oldest guid in the db as starting point, find episodes on the RSS feed not in the db
    new_guids = set([ep["guid"] for ep in episodes_all["audioitem_set"][:idx]]) - set(episodes_db)
    new_guids = list(new_guids)

    # get the full episode dictionary for each new guid
    new_eps = []
    for guid in new_guids:
        for episode in episodes_all["audioitem_set"]:
            if episode["guid"] == guid:
                new_eps.append(episode)
    return new_eps

    

In [5]:
import os

def download_audio_file(new_ep, podcast_slug):
    current_dir = os.path.dirname(os.getcwd()) # get parent of current directory
    media_dir = current_dir + "/media"

    link = new_ep.get("audio_link")
    guid = new_ep.get("guid")
    fileroot = f"{media_dir}/{podcast_slug}_{guid}"
    # check for wav file
    if not os.path.exists(f"{fileroot}.wav"):
        # check for mp3 file
        if not os.path.exists(f"{fileroot}.mp3"):
            # download mp3 file
            print(f"Downloading {fileroot}")
            !curl -o '{fileroot + ".mp3"}' -L -J '{link}'
        # convert mp3 to wav
        !ffmpeg -i '{fileroot + ".mp3"}' -vn -acodec pcm_s16le -ar 16000 -ac 1 '{fileroot + ".wav"}'
        print(f"Downloaded {fileroot}")
    else:
        print(f"File {fileroot} already exists")


    filepath = fileroot + ".wav"
    return filepath

In [6]:
from transcribe_file import get_transcription
import sentence_splitter
import spacy

# update podcasts that are already in the database with new episodes
for podcast in podcasts:
    slug = podcast["slug"]
    print(slug)
    new_eps = get_new_episodes(podcast)
    files = []
    for ep in new_eps:
        print(ep.get("title"))
        # get file
        file = download_audio_file(ep, slug)
        # run transcription
        lang = podcast["language"][0:2]
        script = get_transcription(file, language=lang if lang != "nb" else "no", model_size="large")

        # post episode to api
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/episodes/", json=ep)
        print(res.status_code)

        # post transcription to api
        transcription_dict = script["transcription"]
        guid = "_".join(file.split("/")[-1].split("_")[1:]).split(".")[0]
        transcription_dict["guid"] = guid
        res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
        print(res.status_code)

        # post whisper default segmentation
        segmentation_dict = script["segmentation"]
        trans_uuid = res.json().get("uuid")
        segmentation_dict["uuid"] = trans_uuid
        res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict)
        print(res.status_code)

        # use spacy to split the text into sentences
        try :
            utterances = sentence_splitter.sentence_splitter(transcription_dict, "en_core_web_lg" if lang == "en" else "nb_core_news_lg")

            segmentation_dict_spacy = {
                "uuid": trans_uuid,
                "name": "spaCy",
                "segmentor": {"name": "spaCy", "version": spacy.__version__},
                "utterance_set": [utterance for utterance in utterances if utterance.get("text") != ""]
            }

            res = requests.post(f"{SERVER_URL}/api/podcasts/{slug}/{guid}/utterances/", json=segmentation_dict_spacy)
            print("spaCy", res.status_code)
        except:
            print("spaCy failed")



/home/adamj/anaconda3/envs/whspr/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


verdict-with-ted-cruz
ta-kommandoen-med-geir-aker
misjonen-med-antonsen-og-golden
leger-om-livet
norsken-svensken-og-dansken
burde-vrt-pensum
huberman-lab
stuff-you-should-know
Carbon Monoxide: Please Just Listen Anyway
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:--  0:00:01 --:--:--     0
100   364  100   364    0     0    188      0  0:00:01  0:00:01 --:--:--   955
  0     0    0     0    0     0      0      0 --:--:--  0:00:02 --:--:--     0
100   975  100   975    0     0    333      0  0:00:02  0:00:02 --:--:--  2859
100 46.7M  100 46.7M    0     0  5088k      0  0:00:09  0:00:09 --:--:-- 9879k
ffmpeg version 4.3 Copyright (c) 2000-2020 the FFmpeg developers
  built with gcc 7.3.0 (crosstool-NG 1.23.0.449-a04d0)
  configuration: --prefix=/opt/conda/conda-bld/ffmpeg_1597178665428/_h_env_placehold_placehold_placehold_pl

In [ ]:
# RSS for 20 podcasts
RSS = {
    # Verdict with Ted Cruz
    "https://www.omnycontent.com/d/playlist/e73c998e-6e60-432f-8610-ae210140c5b1/2bee9419-43de-46ce-8996-af2a01167517/84cf551f-a33b-41d8-b112-af2a01167541/podcast.rss": "en",
    # Ta Kommandoen med Geir Aker
    "https://feeds.acast.com/public/shows/63f77218886da700110bf9f9": "no",
    # Misjonen med Antonsen og Golden
    "https://smartpod.no/feed/misjonen": "no",
    # Leger om livet
    "https://feeds.acast.com/public/shows/5f883c4673174a1b3a21b64b": "no",
    # Norsken, svensken og dansken
    "https://podkast.nrk.no/program/norsken_svensken_og_dansken.rss": "no",
    # Burde vært pensum
    "https://podkast.nrk.no/program/burde_vaert_pensum.rss": "no",
    # Huberman Lab
    "https://feeds.megaphone.fm/hubermanlab": "en",
    # Stuff You Should Know
    "https://www.omnycontent.com/d/playlist/e73c998e-6e60-432f-8610-ae210140c5b1/a91018a4-ea4f-4130-bf55-ae270180c327/44710ecc-10bb-48d1-93c7-ae270180c33e/podcast.rss": "en",
    # The Ben Shapiro Show
    "https://feeds.megaphone.fm/WWO8086402096": "en",
    # The Megyn Kelly Show  
    "https://feeds.simplecast.com/RV1USAfC": "en",
    # Pivot
    "https://feeds.megaphone.fm/pivot": "en",
    # The Daily
    "https://feeds.simplecast.com/54nAGcIl": "en",
    # The Ezra Klein Show
    "https://feeds.simplecast.com/82FI35Px": "en",
    # Lex Fridman Podcast
    "https://lexfridman.com/feed/podcast/": "en",
    # Checks and Balance from The Economist
    "https://rss.acast.com/checksandbalance": "en",
    # Money Talks from The Economist
    "https://rss.acast.com/theeconomistmoneytalks": "en",
    # The Economist Asks
    "https://rss.acast.com/theeconomistasks": "en",
    # The Ramsey Show
    "https://feeds.megaphone.fm/RM4031649020": "en",
    # Dateline NBC
    "https://podcastfeeds.nbcnews.com/HL4TzgYC": "en",
    # Pod Save America
    "https://feeds.simplecast.com/dxZsm5kX": "en",
    # Freakonomics Radio
    "https://feeds.simplecast.com/Y8lFbOT4": "en",

}

In [ ]:
# Add each podcast in RSS to database via API

for podcast in RSS:
  print(podcast)
  res = requests.post("http://127.0.0.1:8008/api/podcasts/", json={
    "rss": podcast
  })
  print(res.status_code)